In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "8"

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Visible device count:", torch.cuda.device_count())
print("Using GPU:", torch.cuda.get_device_name(0))

CUDA available: True
Visible device count: 1
Using GPU: Tesla V100-PCIE-32GB


In [3]:
import numpy
import mmcv
import mmengine
import mmpose

print("numpy:", numpy.__version__)
print("mmcv:", mmcv.__version__)
print("mmengine:", mmengine.__version__)
print("mmpose:", mmpose.__version__)

numpy: 1.26.4
mmcv: 2.1.0
mmengine: 0.10.7
mmpose: 1.3.2


In [4]:
import torch
torch.jit._state.disable()


In [5]:
import torch
import numpy
import mmcv
import mmdet
import mmpose
from mmpose.apis import MMPoseInferencer

print("torch:", torch.__version__)
print("numpy:", numpy.__version__)
print("mmcv:", mmcv.__version__)
print("mmdet:", mmdet.__version__)
print("mmpose:", mmpose.__version__)
print("CUDA:", torch.cuda.is_available())

/home/j-i14a203/.conda/envs/mmpose/lib/python3.10/site-packages/mmengine/utils/package_utils.py:48: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


torch: 2.1.2+cu121
numpy: 1.26.4
mmcv: 2.1.0
mmdet: 3.3.0
mmpose: 1.3.2
CUDA: True


In [6]:
from mmpose.apis import MMPoseInferencer

In [32]:
from mmengine.config import Config

# inferencer 생성

inferencer = MMPoseInferencer(
    pose2d="../checkpoints/rtmpose-l_8xb32-270e_coco-wholebody-384x288.py",
    pose2d_weights="../checkpoints/rtmpose-l_simcc-coco-wholebody_pt-aic-coco_270e-384x288-eaeb96c8_20230125.pth",
    det_model="../checkpoints_det/rtmdet_l_8xb32-300e_coco.py",
    det_weights="../checkpoints_det/rtmdet_l_8xb32-300e_coco_20220719_112030-5a0be7c4.pth",
    device="cuda"
)


# 테스트 이미지 (경로 수정)
img_path = "../data/images_test/test.jpg"

# 추론
result = next(
    inferencer(
        img_path,
        return_vis=False,
        show=False
    )
)

print(result.keys())

Loads checkpoint by local backend from path: ../checkpoints/rtmpose-l_simcc-coco-wholebody_pt-aic-coco_270e-384x288-eaeb96c8_20230125.pth
Loads checkpoint by local backend from path: ../checkpoints_det/rtmdet_l_8xb32-300e_coco_20220719_112030-5a0be7c4.pth
The model and loaded state dict do not match exactly

unexpected key in source state_dict: data_preprocessor.mean, data_preprocessor.std

dict_keys(['visualization', 'predictions'])


In [34]:
preds = result['predictions']
person = preds[0][0]
print(preds)
print('---')
print(person)

[[{'keypoints': [[364.4444580078125, 81.94445037841797], [372.77777099609375, 75.00000762939453], [357.5, 75.00000762939453], [388.0555419921875, 77.77778625488281], [351.9444580078125, 79.16667175292969], [407.5, 113.8888931274414], [367.22222900390625, 122.22222900390625], [440.8333435058594, 152.7777862548828], [350.5555419921875, 162.5], [439.4444580078125, 161.11111450195312], [303.3333435058594, 180.55555725097656], [433.8888854980469, 204.1666717529297], [390.8333435058594, 209.72222900390625], [436.6666564941406, 281.9444274902344], [374.1666564941406, 272.22222900390625], [471.3888854980469, 361.1111145019531], [399.1666564941406, 344.4444274902344], [445.0, 375.0], [457.5, 376.3888854980469], [493.6111145019531, 363.8888854980469], [372.77777099609375, 366.6666564941406], [372.77777099609375, 359.72222900390625], [420.0, 355.5555419921875], [353.3333435058594, 77.77778625488281], [354.72222900390625, 80.55555725097656], [354.72222900390625, 84.72222900390625], [354.7222290039

In [22]:
print(person.keys())

dict_keys(['keypoints', 'keypoint_scores', 'bbox', 'bbox_score'])


In [35]:
keypoints = person['keypoints']
print(len(keypoints))
# (133, 2)


133


In [36]:
KEYPOINT_MAPPING = {
    # COCO 17
    0: 0,    # nose
    1: 1,    # left_eye
    2: 2,    # right_eye
    3: 3,    # left_ear
    4: 4,    # right_ear
    5: 5,    # left_shoulder
    6: 6,    # right_shoulder
    7: 7,    # left_elbow
    8: 8,    # right_elbow
    9: 9,    # left_wrist
    10: 10,  # right_wrist
    11: 11,  # left_hip
    12: 12,  # right_hip
    13: 13,  # left_knee
    14: 14,  # right_knee
    15: 15,  # left_ankle
    16: 16,  # right_ankle

    # Foot extra
    17: 19,  # left_heel
    18: 22,  # right_heel
    19: 17,  # left_big_toe
    20: 20   # right_big_toe
}


In [37]:
import numpy as np

def extract_21_keypoints(mmpose_result):
    """
    return: (21, 3) -> x, y, score
    """
    person = mmpose_result["predictions"][0][0]
    kps = person["keypoints"]          # (133, 2)
    scores = person["keypoint_scores"] # (133,)

    out = np.zeros((21, 3))

    for new_idx, mmpose_idx in KEYPOINT_MAPPING.items():
        out[new_idx, 0] = kps[mmpose_idx][0]
        out[new_idx, 1] = kps[mmpose_idx][1]
        out[new_idx, 2] = scores[mmpose_idx]

    return out


In [38]:
# YOLO 기준 스켈레톤
SKELETON = [
    (5, 7), (7, 9),      # left arm
    (6, 8), (8, 10),     # right arm
    (5, 6),              # shoulders
    (11, 13), (13, 15),  # left leg
    (12, 14), (14, 16),  # right leg
    (11, 12),            # hips
    (15, 17), (17, 19),  # left foot
    (16, 18), (18, 20),  # right foot
]


In [39]:
import cv2

def draw_keypoints(img, keypoints, score_thr=0.3):
    img = img.copy()

    # draw points
    for i, (x, y, s) in enumerate(keypoints):
        if s > score_thr:
            cv2.circle(img, (int(x), int(y)), 4, (0, 255, 0), -1)
            cv2.putText(
                img, str(i),
                (int(x)+3, int(y)-3),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.4, (255, 0, 0), 1
            )

    # draw skeleton
    for a, b in SKELETON:
        if keypoints[a][2] > score_thr and keypoints[b][2] > score_thr:
            pt1 = tuple(map(int, keypoints[a][:2]))
            pt2 = tuple(map(int, keypoints[b][:2]))
            cv2.line(img, pt1, pt2, (0, 255, 255), 2)

    return img


In [40]:
img = cv2.imread(img_path)

kps21 = extract_21_keypoints(result)

vis = draw_keypoints(img, kps21)

cv2.imwrite("vis_21_keypoints.jpg", vis)

True